# Empirical Pipeline: Phantom Labels, NN Training, MM Backtest

This notebook implements the full empirical pipeline:

1. **Part 1: Phantom Label Generation** — Extract phantom-fill labels from empirical KGHM order-flow data
2. **Part 2: Neural Network Training** — Train a fill-probability MLP on the empirical labels
3. **Part 3: Market Maker Backtest** — Backtest ergodic and NN-based market makers on empirical data

In [ ]:
%pip install -e ..

from __future__ import annotations

import functools
import json
import os
import random
import sqlite3
import subprocess
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import curve_fit, minimize_scalar

try:
    import pyarrow  # noqa: F401
except ImportError as e:
    raise ImportError(
        "Install pyarrow to write parquet (e.g. `%pip install pyarrow`)."
    ) from e

from research_core.classes import (
    PhantomLabelConfig,
    PhantomLabeller,
    list_day_keys_hdf,
    run_full_extraction,
    write_day_parquet,
    write_feature_schema,
    write_manifest,
)
from research_core.classes.helpers import project_root, resolve_data_path

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [ ]:
# --- edit paths below; extraction flags default to auto for 40 LOB levels ---

N_BOOK_LEVELS_TARGET = 0   # no per-level depth features; queue_ahead suffices
N_QUEUE_LEVELS = 40        # same-side levels extracted for queue_ahead (L0..L39)

def _orders_has_depth_level(db: Path, level_last: int) -> bool:
    """True if ``orders`` includes ``bid_depth_L{level_last}``."""
    if not db.is_file():
        return False
    col = f"bid_depth_L{level_last}"
    with sqlite3.connect(str(db)) as conn:
        cols = [r[1] for r in conn.execute("PRAGMA table_info(orders)").fetchall()]
    return col in cols


# HDF5 sources (used when extraction runs). Adjust to your WSE layout.
ASSET = "KGHM"
# WSELOB-2017 layout: orders = ``{ASSET}_lob_2017_zlib.h5``, trades = ``{ASSET}_trades_2017_zlib.h5``
ORDERS_H5 = project_root() / "data" / "WSELOB-2017" / "orders" / f"{ASSET}_lob_2017_zlib.h5"
TRADES_H5 = project_root() / "data" / "WSELOB-2017" / "trades" / f"{ASSET}_trades_2017_zlib.h5"

DB_RELNAME = "KGHM_order_flow.sqlite"  # resolved under data_dir()
DB_PATH = resolve_data_path(DB_RELNAME)

# Auto: extract when DB is missing or lacks top depth column (old 5-level extract).
# Override with True/False if you want to force refresh or skip HDF5 entirely.
RUN_EXTRACTION: Optional[bool] = None
FORCE_EXTRACTION: Optional[bool] = None

_has_top = _orders_has_depth_level(DB_PATH, N_QUEUE_LEVELS - 1)
_auto_run = (not DB_PATH.is_file()) or (not _has_top)
if RUN_EXTRACTION is None:
    RUN_EXTRACTION = _auto_run
if FORCE_EXTRACTION is None:
    FORCE_EXTRACTION = DB_PATH.is_file() and _auto_run

print(
    f"LOB check: bid_depth_L{N_QUEUE_LEVELS - 1} "
    f"{'OK' if _has_top else 'MISSING'} in {DB_PATH.name} → "
    f"RUN_EXTRACTION={RUN_EXTRACTION}, FORCE_EXTRACTION={FORCE_EXTRACTION}"
)

# None = all day keys found in the orders HDF5
EXTRACTION_DAY_KEYS: Optional[List[str]] = None

# Phantom parquet output directory (relative name => data_dir() / ...)
PHANTOM_OUT_REL = "phantom_labels"
PHANTOM_OUT_DIR = resolve_data_path(PHANTOM_OUT_REL)

# None = every day present in SQLite; else explicit list
PHANTOM_DAYS: Optional[List[str]] = None
SKIP_EXISTING_PARQUET = True

# Debug: cap labelling timestamps per day (None = full day)
MAX_ROWS: Optional[int] = None

PHANTOM_CFG = PhantomLabelConfig(
    tau=1.0,
    cadence="bbo_change",
    delta_lo=-0.50,
    max_delta=2.0,
    tick_size=0.05,
    phantom_size=1,
    vol_halflife=50,
    n_book_levels=N_BOOK_LEVELS_TARGET,
    n_queue_levels=N_QUEUE_LEVELS,
)

In [ ]:
# Stage A — HDF5 → SQLite (optional)
if RUN_EXTRACTION:
    for p in (ORDERS_H5, TRADES_H5):
        if not Path(p).is_file():
            raise FileNotFoundError(f"Missing HDF5: {p}")
    day_keys = EXTRACTION_DAY_KEYS
    if day_keys is None:
        day_keys = list_day_keys_hdf(Path(ORDERS_H5))
    print(f"Extracting {len(day_keys)} day(s) → {DB_PATH.name}")
    run_full_extraction(
        ASSET,
        Path(ORDERS_H5),
        Path(TRADES_H5),
        Path(DB_PATH),
        market_open="09:00:00",
        market_close="16:50:00",
        tick_size=0.05,
        force=FORCE_EXTRACTION,
        day_keys=day_keys,
        depth_levels=N_QUEUE_LEVELS,
    )
else:
    if DB_PATH.is_file() and not _orders_has_depth_level(DB_PATH, N_QUEUE_LEVELS - 1):
        print(
            f"RUN_EXTRACTION=False but `orders` lacks bid_depth_L{N_QUEUE_LEVELS - 1}+ — "
            "phantom will zero-fill missing depth columns. Set RUN_EXTRACTION=True to rebuild."
        )
    else:
        print("RUN_EXTRACTION=False — skipping HDF5 → SQLite.")

In [ ]:
# Stage B — SQLite → phantom parquet + schema + manifest


def _git_commit() -> Optional[str]:
    try:
        out = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=str(project_root()),
            check=True,
            capture_output=True,
            text=True,
            timeout=5,
        )
        return out.stdout.strip()
    except Exception:
        return None


def _infer_symbol(db_path: str) -> str:
    name = Path(db_path).stem
    head = name.split("_", 1)[0]
    return head.upper() if head else "UNKNOWN"


def _resolve_phantom_days(
    labeller: PhantomLabeller, days: Optional[Sequence[str]],
) -> List[str]:
    available = labeller.list_days()
    if not days:
        return list(available)
    missing = [d for d in days if d not in available]
    if missing:
        raise ValueError(
            f"requested days not in DB: {missing[:5]} "
            f"(DB has {len(available)} days; sample: {available[:3]})"
        )
    seen: set[str] = set()
    ordered: List[str] = []
    for d in days:
        if d not in seen:
            ordered.append(d)
            seen.add(d)
    return ordered


if not DB_PATH.is_file():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

labeller = PhantomLabeller(DB_PATH, config=PHANTOM_CFG, hawkes=True)
days_to_run = _resolve_phantom_days(labeller, PHANTOM_DAYS)
symbol = _infer_symbol(str(DB_PATH))

if SKIP_EXISTING_PARQUET:
    before = len(days_to_run)
    days_to_run = [
        d for d in days_to_run
        if not (PHANTOM_OUT_DIR / f"{symbol}_{d}.parquet").exists()
    ]
    print(f"SKIP_EXISTING_PARQUET: {before - len(days_to_run)} skipped, {len(days_to_run)} to run.")
    if not days_to_run:
        print("Nothing to do (all parquets exist).")

if days_to_run:
    print(
        f"Labelling {len(days_to_run)} day(s) | tau={PHANTOM_CFG.tau}s "
        f"cadence={PHANTOM_CFG.cadence} | DB={DB_PATH} | out={PHANTOM_OUT_DIR}"
    )
    print(f"  features per t: {len(labeller.feature_names)}", flush=True)

PHANTOM_OUT_DIR.mkdir(parents=True, exist_ok=True)

successful_days: List[str] = []
failed_days: List[tuple[str, str]] = []
total_rows = 0
t0 = time.time()

for day_key in days_to_run:
    t_day = time.time()
    try:
        df = labeller.label_day(day_key, max_rows=MAX_ROWS)
        elapsed = time.time() - t_day
        if df.empty:
            print(f"  {day_key}: empty (no usable rows) [{elapsed:.1f}s]")
            continue
        out_path = PHANTOM_OUT_DIR / f"{symbol}_{day_key}.parquet"
        n_rows = write_day_parquet(df, out_path)
        successful_days.append(day_key)
        total_rows += n_rows
        print(f"  {day_key}: {n_rows:,} rows → {out_path.name} [{elapsed:.1f}s]", flush=True)
    except Exception as exc:
        import traceback

        err = f"{exc!r}\n{traceback.format_exc()}"
        failed_days.append((day_key, err))
        print(f"  {day_key}: FAILED — {err.splitlines()[0]}", flush=True)

runtime = time.time() - t0
print(
    f"\nDone in {runtime:.1f}s | success={len(successful_days)} "
    f"failed={len(failed_days)} total_rows={total_rows:,}"
)

if successful_days:
    schema = labeller.feature_schema()
    write_feature_schema(schema, PHANTOM_OUT_DIR)
    write_manifest(
        PHANTOM_OUT_DIR,
        db_path=str(DB_PATH),
        days_processed=successful_days,
        config=PHANTOM_CFG,
        total_rows=total_rows,
        runtime_seconds=runtime,
        git_commit=_git_commit(),
        hawkes_n_kernels=labeller._n_kernels,
    )
    print(f"Wrote feature_schema.json and manifest.json → {PHANTOM_OUT_DIR}")
if failed_days:
    print("Failed days (first line only):")
    for d, e in failed_days:
        print(f"  {d}: {e.splitlines()[0]}")

In [ ]:
# Sanity — quick read-back (run after Stage B in the same kernel)
if not globals().get("successful_days"):
    print("Run Stage B first (or no days succeeded). Skipping read-back.")
elif not globals().get("labeller"):
    print("labeller not defined — run Stage B first.")
else:
    _stem = Path(DB_PATH).stem
    _sym = _stem.split("_", 1)[0].upper() if _stem else "UNKNOWN"
    sample = PHANTOM_OUT_DIR / f"{_sym}_{successful_days[0]}.parquet"
    chk = pd.read_parquet(sample)
    print("sample file:", sample)
    print("rows:", len(chk), "| first columns:", list(chk.columns[:8]), "...")
    print("len(feature_names):", len(labeller.feature_names))
    schema_path = PHANTOM_OUT_DIR / "feature_schema.json"
    if schema_path.is_file():
        meta = json.loads(schema_path.read_text())
        print("schema n_per_t_features:", meta.get("n_per_t_features"))

## Part 2: Neural Network Training

In [ ]:
# Empirical phantom labels (phantom_training_data.ipynb → data/phantom_labels/).
phantom_dir = resolve_data_path("phantom_labels")
symbol: Optional[str] = "KGHM"  # files are KGHM_<day>.parquet
max_rows_per_day: Optional[int] =  None
batch_size = 65536
epochs = 40
lr = 3e-4
hidden = [64, 32]
dropout = 0.15
seed = 42
pos_weight: Optional[float] = None  # e.g. 2.0 if fills are rare

# queue_ahead feature transform.  "log1p" applies np.log1p before
# normalisation so the model has better resolution near BBO where
# queue_ahead is small.  None keeps the raw value (legacy behaviour).
# Recorded in the checkpoint so inference auto-selects the matching
# transform.
queue_transform: Optional[str] = "log1p"

# Checkpoint dir for mm_backtest Track 2 (NN_EMP_CKPT → phantom_models/).
out_dir = resolve_data_path("phantom_models")

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
schema_path = phantom_dir / "feature_schema.json"
manifest_path = phantom_dir / "manifest.json"
assert schema_path.exists(), f"Missing {schema_path}"
assert manifest_path.exists(), f"Missing {manifest_path}"

with open(schema_path) as f:
    schema = json.load(f)
with open(manifest_path) as f:
    manifest = json.load(f)

# recent_vol training definition (recorded into the checkpoint so the backtest
# reconstructs the SAME feature at inference; see mm_backtest_parallel auto mode).
manifest_config = manifest.get("config", {})
vol_mode = manifest_config.get("vol_mode", "ewma_event")
vol_window_s = manifest_config.get("vol_window_s", None)
vol_sample_dt_s = manifest_config.get("vol_sample_dt_s", None)
if vol_mode == "realized_time":
    vol_detail = f" (window={vol_window_s}s, dt={vol_sample_dt_s}s)"
else:
    vol_detail = ""
print(f"recent_vol mode: {vol_mode}{vol_detail}")

tick_size = schema["tick_size"]
tau = schema["tau"]
n_delta = schema["delta_grid"]["n_points"]

feat_cols = [c for c, info in schema["columns"].items() if info["role"] == "feature"]
input_cols_schema = [c for c, info in schema["columns"].items() if info["role"] == "input"]

all_days = manifest["days_processed"]
print(f"Total trading days in manifest: {len(all_days)}")

# Auto-detect parquet prefix if symbol is None ({symbol}_{day}.parquet).
if symbol is None:
    detected_prefix = None
    for day_key in all_days:
        matching_files = list(phantom_dir.glob(f"*_{day_key}.parquet"))
        if matching_files:
            detected_prefix = matching_files[0].name[: -(len(day_key) + len(".parquet") + 1)]
            break
    if detected_prefix is None:
        raise FileNotFoundError(
            f"No parquet matching *_<day>.parquet in {phantom_dir}; "
            "set symbol explicitly in the config cell."
        )
    symbol = detected_prefix
print(f"Parquet prefix (symbol): {symbol!r}")

# Chronological trading-day split (NO shuffling across days => no temporal
# leakage).  Fractions adapt to however many days are in the manifest.
train_frac, val_frac = 0.80, 0.10
n_days = len(all_days)
assert n_days >= 3, f"Need >=3 days/chunks for a train/val/test split, got {n_days}"
n_train = min(max(1, round(n_days * train_frac)), n_days - 2)
n_val = min(max(1, round(n_days * val_frac)), n_days - n_train - 1)

train_days = all_days[:n_train]
val_days = all_days[n_train : n_train + n_val]
test_days = all_days[n_train + n_val :]

print(f"\nSplit: train={len(train_days)}, val={len(val_days)}, test={len(test_days)}")
print(f"  Train: {train_days[0]} .. {train_days[-1]}")
print(f"  Val:   {val_days[0]} .. {val_days[-1]}")
print(f"  Test:  {test_days[0]} .. {test_days[-1]}")

def day_to_path(day_key: str) -> Path:
    return phantom_dir / f"{symbol}_{day_key}.parquet"

train_files = [day_to_path(day_key) for day_key in train_days]
val_files = [day_to_path(day_key) for day_key in val_days]
test_files = [day_to_path(day_key) for day_key in test_days]

for parquet_path in train_files + val_files + test_files:
    assert parquet_path.exists(), f"Missing parquet: {parquet_path}"

In [ ]:
norm_cols = feat_cols + ["delta", "queue_ahead"]
n_norm = len(norm_cols)
queue_ahead_idx = norm_cols.index("queue_ahead")

running_sum = np.zeros(n_norm, dtype=np.float64)
running_sq = np.zeros(n_norm, dtype=np.float64)
running_n = 0

t0 = time.time()
for file_idx, parquet_path in enumerate(train_files):
    day_data = pd.read_parquet(parquet_path, columns=norm_cols)
    if max_rows_per_day is not None and len(day_data) > max_rows_per_day:
        day_data = day_data.sample(n=max_rows_per_day, random_state=seed)
    vals = day_data[norm_cols].values.astype(np.float64)
    if queue_transform == "log1p":
        vals[:, queue_ahead_idx] = np.log1p(vals[:, queue_ahead_idx])
    running_sum += vals.sum(axis=0)
    running_sq += (vals ** 2).sum(axis=0)
    running_n += len(vals)
    if (file_idx + 1) % max(1, len(train_files) // 5) == 0:
        print(f"  Stats pass: {file_idx+1}/{len(train_files)} files, {running_n:,} rows")

feat_mean = (running_sum / running_n).astype(np.float32)
feat_std = np.sqrt(running_sq / running_n - feat_mean.astype(np.float64) ** 2).astype(np.float32)
feat_std[feat_std < 1e-8] = 1.0

print(f"\nFeature stats computed in {time.time() - t0:.1f}s over {running_n:,} rows")
print(f"  queue_transform: {queue_transform or 'None (raw)'}")

In [ ]:
def load_day_tensors(
    path: Path,
    feat_cols: List[str],
    feat_mean: np.ndarray,
    feat_std: np.ndarray,
    max_rows: Optional[int] = None,
    queue_transform: Optional[str] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Load one day's parquet, normalize, shuffle, return (X, y) tensors."""
    day_data = pd.read_parquet(path)
    if max_rows is not None and len(day_data) > max_rows:
        day_data = day_data.sample(n=max_rows, random_state=None)
    day_data = day_data.sample(frac=1.0)

    norm_cols = feat_cols + ["delta", "queue_ahead"]
    norm_vals = day_data[norm_cols].values.astype(np.float32)
    if queue_transform == "log1p":
        queue_ahead_idx = norm_cols.index("queue_ahead")
        norm_vals[:, queue_ahead_idx] = np.log1p(norm_vals[:, queue_ahead_idx])
    norm_vals = (norm_vals - feat_mean) / feat_std

    side = (day_data["side"].values.astype(np.float32) - 1.0).reshape(-1, 1)

    X = np.hstack([norm_vals, side])
    y = day_data["label"].values.astype(np.float32)

    return torch.from_numpy(X), torch.from_numpy(y)

In [ ]:
class FillProbMLP(nn.Module):
    def __init__(self, in_dim: int, hidden: List[int], dropout: float = 0.0):
        super().__init__()
        layers: list = []
        prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


in_dim = len(feat_cols) + 3  # features + delta + queue_ahead + side
model = FillProbMLP(in_dim, hidden, dropout=dropout).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
if pos_weight:
    pos_weight_tensor = torch.tensor([pos_weight], device=device)
else:
    pos_weight_tensor = None
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

train_losses: list[float] = []
val_losses: list[float] = []
val_briers: list[float] = []
best_val_loss = float("inf")
best_state: Optional[dict] = None

t_start = time.time()
for epoch in range(epochs):
    t_epoch = time.time()

    # --- Train ---
    model.train()
    epoch_loss, epoch_n = 0.0, 0
    for parquet_path in train_files:
        X, y = load_day_tensors(
            parquet_path, feat_cols, feat_mean, feat_std,
            max_rows_per_day, queue_transform=queue_transform,
        )
        X, y = X.to(device), y.to(device)
        for i in range(0, len(X), batch_size):
            xb, yb = X[i : i + batch_size], y[i : i + batch_size]
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
            epoch_n += len(xb)
    train_losses.append(epoch_loss / epoch_n)

    # --- Val ---
    model.eval()
    val_loss_sum, val_brier_sum, val_n = 0.0, 0.0, 0
    with torch.no_grad():
        for parquet_path in val_files:
            X, y = load_day_tensors(
                parquet_path, feat_cols, feat_mean, feat_std,
                None, queue_transform=queue_transform,
            )
            X, y = X.to(device), y.to(device)
            logits = model(X)
            val_loss_sum += nn.functional.binary_cross_entropy_with_logits(
                logits, y, reduction="sum"
            ).item()
            probs = torch.sigmoid(logits)
            val_brier_sum += ((probs - y) ** 2).sum().item()
            val_n += len(y)
    val_loss = val_loss_sum / val_n
    val_brier = val_brier_sum / val_n
    val_losses.append(val_loss)
    val_briers.append(val_brier)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    scheduler.step()

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"| train_loss={train_losses[-1]:.5f} "
        f"| val_loss={val_loss:.5f} "
        f"| val_brier={val_brier:.5f} "
        f"| lr={scheduler.get_last_lr()[0]:.2e} "
        f"| {time.time() - t_epoch:.1f}s"
    )

print(f"\nBest val loss: {best_val_loss:.5f}  (total {time.time() - t_start:.0f}s)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
epochs_x = range(1, len(train_losses) + 1)
ax.plot(epochs_x, train_losses, "o-", label="Train loss")
ax.plot(epochs_x, val_losses, "s-", label="Val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE loss")
ax.set_title("Training curves")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Restore best checkpoint ---
assert best_state is not None, "No best state saved — training may have failed"
model.load_state_dict(best_state)
model.eval()

# --- Temperature scaling (post-hoc calibration on val set) ---

val_logits_all = []
val_labels_all = []
with torch.no_grad():
    for parquet_path in val_files:
        X, y = load_day_tensors(
            parquet_path, feat_cols, feat_mean, feat_std,
            None, queue_transform=queue_transform,
        )
        X, y = X.to(device), y.to(device)
        logits = model(X)
        val_logits_all.append(logits)
        val_labels_all.append(y)

val_logits_cat = torch.cat(val_logits_all)
val_labels_cat = torch.cat(val_labels_all)

def nll_with_temp(temp):
    scaled = val_logits_cat / temp
    return F.binary_cross_entropy_with_logits(scaled, val_labels_cat).item()

temp_result = minimize_scalar(nll_with_temp, bounds=(0.1, 10.0), method='bounded')
temperature = temp_result.x
print(f"Optimal temperature: {temperature:.4f}")
print(f"  Val NLL before temp scaling: {nll_with_temp(1.0):.5f}")
print(f"  Val NLL after  temp scaling: {nll_with_temp(temperature):.5f}")

# --- Collect test predictions (temperature-scaled) ---
all_probs, all_labels, all_delta, all_queue, all_side = [], [], [], [], []

with torch.no_grad():
    for parquet_path in test_files:
        day_data = pd.read_parquet(parquet_path)
        norm_cols = feat_cols + ["delta", "queue_ahead"]
        norm_vals = day_data[norm_cols].values.astype(np.float32)
        if queue_transform == "log1p":
            queue_ahead_idx = norm_cols.index("queue_ahead")
            norm_vals[:, queue_ahead_idx] = np.log1p(norm_vals[:, queue_ahead_idx])
        norm_vals = (norm_vals - feat_mean) / feat_std
        side_enc = (day_data["side"].values.astype(np.float32) - 1.0).reshape(-1, 1)
        features = np.hstack([norm_vals, side_enc])
        features_tensor = torch.from_numpy(features).to(device)
        logits = model(features_tensor)
        probs = torch.sigmoid(logits / temperature).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(day_data["label"].values.astype(np.float32))
        all_delta.append(day_data["delta"].values.astype(np.float32))
        all_queue.append(day_data["queue_ahead"].values.astype(np.float32))
        all_side.append(day_data["side"].values.astype(np.int8))

probs_test = np.concatenate(all_probs)
labels_test = np.concatenate(all_labels)
delta_test = np.concatenate(all_delta)
queue_test = np.concatenate(all_queue)
side_test = np.concatenate(all_side)

# --- NN test metrics ---
eps = 1e-7
nn_bce = -np.mean(
    labels_test * np.log(probs_test + eps)
    + (1 - labels_test) * np.log(1 - probs_test + eps)
)
nn_brier = np.mean((probs_test - labels_test) ** 2)

# --- Constant baseline: predict mean(y_train) ---
train_mean_y = 0.0
train_total = 0
for parquet_path in train_files:
    day_data = pd.read_parquet(parquet_path, columns=["label"])
    if max_rows_per_day is not None and len(day_data) > max_rows_per_day:
        day_data = day_data.sample(n=max_rows_per_day, random_state=seed)
    train_mean_y += day_data["label"].sum()
    train_total += len(day_data)
train_mean_y /= train_total

const_bce = -np.mean(
    labels_test * np.log(train_mean_y + eps)
    + (1 - labels_test) * np.log(1 - train_mean_y + eps)
)
const_brier = np.mean((train_mean_y - labels_test) ** 2)

# --- Queue-only linear baseline ---
# Collect train data for the linear model
lin_X_parts, lin_y_parts = [], []
for parquet_path in train_files:
    day_data = pd.read_parquet(parquet_path, columns=["delta", "queue_ahead", "side", "label"])
    if max_rows_per_day is not None and len(day_data) > max_rows_per_day:
        day_data = day_data.sample(n=max_rows_per_day, random_state=seed)
    linear_features = np.column_stack([
        day_data["delta"].values.astype(np.float32),
        day_data["queue_ahead"].values.astype(np.float32),
        (day_data["side"].values.astype(np.float32) - 1.0),
    ])
    lin_X_parts.append(linear_features)
    lin_y_parts.append(day_data["label"].values.astype(np.float32))

lin_X_train = torch.from_numpy(np.concatenate(lin_X_parts)).to(device)
lin_y_train = torch.from_numpy(np.concatenate(lin_y_parts)).to(device)

lin_model = nn.Linear(3, 1).to(device)
lin_opt = torch.optim.Adam(lin_model.parameters(), lr=1e-2)
lin_crit = nn.BCEWithLogitsLoss()

for _ in range(50):
    for i in range(0, len(lin_X_train), batch_size):
        xb = lin_X_train[i : i + batch_size]
        yb = lin_y_train[i : i + batch_size]
        logit = lin_model(xb).squeeze(-1)
        loss = lin_crit(logit, yb)
        lin_opt.zero_grad()
        loss.backward()
        lin_opt.step()

lin_X_test = torch.from_numpy(
    np.column_stack([delta_test, queue_test, side_test.astype(np.float32) - 1.0])
).to(device)

with torch.no_grad():
    lin_probs = torch.sigmoid(lin_model(lin_X_test).squeeze(-1)).cpu().numpy()

lin_bce = -np.mean(
    labels_test * np.log(lin_probs + eps)
    + (1 - labels_test) * np.log(1 - lin_probs + eps)
)
lin_brier = np.mean((lin_probs - labels_test) ** 2)

# --- Exponential-in-delta baseline ---
def exp_fill(delta, amplitude, decay_rate):
    return amplitude * np.exp(-decay_rate * delta)

exp_delta_parts, exp_label_parts = [], []
for parquet_path in train_files:
    day_data = pd.read_parquet(parquet_path, columns=["delta", "label"])
    if max_rows_per_day is not None and len(day_data) > max_rows_per_day:
        day_data = day_data.sample(n=max_rows_per_day, random_state=seed)
    exp_delta_parts.append(day_data["delta"].values.astype(np.float64))
    exp_label_parts.append(day_data["label"].values.astype(np.float64))
exp_delta_train = np.concatenate(exp_delta_parts)
exp_label_train = np.concatenate(exp_label_parts)

positive_delta_mask = exp_delta_train > 0
try:
    exp_params, _ = curve_fit(
        exp_fill, exp_delta_train[positive_delta_mask], exp_label_train[positive_delta_mask],
        p0=[1.0, 1.0], maxfev=10000, bounds=([0, 0], [np.inf, np.inf]),
    )
    exp_probs = np.clip(exp_fill(delta_test.astype(np.float64), *exp_params), eps, 1 - eps)
    exp_bce = -np.mean(
        labels_test * np.log(exp_probs)
        + (1 - labels_test) * np.log(1 - exp_probs)
    )
    exp_brier = np.mean((exp_probs - labels_test) ** 2)
except RuntimeError:
    exp_bce = exp_brier = float("nan")

print(f"{'Model':<20s} {'BCE':>10s} {'Brier':>10s}")
print("-" * 42)
print(f"{'Constant baseline':<20s} {const_bce:10.5f} {const_brier:10.5f}")
print(f"{'Queue-only linear':<20s} {lin_bce:10.5f} {lin_brier:10.5f}")
print(f"{'FillProbMLP':<20s} {nn_bce:10.5f} {nn_brier:10.5f}")
print(f"{'Exponential (delta)':<20s} {exp_bce:10.5f} {exp_brier:10.5f}")
print(f"\nTrain fill rate: {train_mean_y:.4f}")
print(f"Test  fill rate: {labels_test.mean():.4f}")
print(f"Test  n_rows:    {len(labels_test):,}")

In [ ]:
# --- Save checkpoint for downstream use by the backtest ---
out_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = out_dir / "fillprob_mlp_best.pt"

ckpt = {
    "model_state_dict": best_state,
    "feat_cols": feat_cols,
    "feat_mean": feat_mean.tolist(),
    "feat_std": feat_std.tolist(),
    "hidden": hidden,
    "dropout": dropout,
    "in_dim": in_dim,
    "temperature": temperature,
    "queue_transform": queue_transform,
    "tick_size": tick_size,
    "tau": tau,
    "n_delta": n_delta,
    "vol_mode": vol_mode,
    "vol_window_s": vol_window_s,
    "vol_sample_dt_s": vol_sample_dt_s,
    "train_days": train_days,
    "val_days": val_days,
    "test_days": test_days,
    "best_val_loss": best_val_loss,
}
torch.save(ckpt, ckpt_path)
print(f"Saved checkpoint → {ckpt_path}")

## Part 3: Market Maker Backtest

In [ ]:
from research_core.classes import (
    MMBacktester, ErgodicMM, NumericalErgodicMM, HawkesFilter,
)
from research_core.classes.helpers import list_day_keys_from_sqlite
from research_core.classes.mm_backtest_parallel import (
    run_ergodic_day_all_gammas, run_numerical_exp_day_all_gammas,
    run_nn_ergodic_day_all_gammas, plan_8h_segments, get_day_span_s,
    run_multi_segment_ergodic_sweep, run_multi_segment_gamma_sweep,
    run_segment_sweep_pair, run_ergodic_segment_gammas,
    single_multivariate_hawkes_factory, triple_quintet44_hawkes_factory,
    _load_nn_bundle, _make_numerical_nn_agent,
)
from research_core.classes.mm_env_diag import (
    quote_vol_decomposition_diag, inventory_path_from_agent,
    inventory_path_metrics, hawkes_spectral_radius, quote_vol_ratio_diag,
    run_env_diag_bundle,
)
from research_core.classes.simulate import SimulateFast

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True})

# --- Paths & replay defaults ---
emp_db = resolve_data_path("KGHM_order_flow.sqlite")
sim_tick = 0.05
bt_size = 1

emp_day = "d20170110"

sweep_gammas = [
    0.0001, 0.0002, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05,
]
smoke_gammas = [0.01, 0.05]
plot_gamma = 0.01
cmp_gamma = 0.05

solver_kw = dict(
    solver_tick=sim_tick / 2,
    max_delta=2.0,
    delta_lo=0.0,
    max_iter=1000,
    tol=1e-3,
    poisson_tau=1.0,
)
match_solver_kw = dict(
    solver_tick=0.00001,
    max_delta=0.2,
    delta_lo=0.0,
    max_iter=10_000,
    tol=1e-4,
    poisson_tau=1.0,
)

nn_emp_ckpt = str(resolve_data_path("phantom_models/fillprob_mlp_best.pt"))

# Vol winner from vol_estimator_selection.ipynb (rv_300s → realized_time grid RV)
erg_vol_kw = dict(
    vol_mode="realized_time",
    rv_window_s=300.0,
    rv_sample_dt_s=1.0,
    vol_floor=1e-8,
)
erg_vol_mode = erg_vol_kw["vol_mode"]  # shorthand for cells that set vol_mode only

# Multi-segment plan (§3a, §7, §10)
segment_sweep_n = 10
segment_sweep_hours = 8.0

# Investigation cells: use enough 8h windows for conclusion-level diagnostics.
# The old two-window seg0/seg1 views are useful smoke tests, but too small for claims.
emp_diag_skip_first = 6
diag_n_segments = 30
env_diag_n_segments = 30
ab_diag_n_segments = 20
layer_c_n_segments = 30

# §3a loss diagnostic
diag_gamma = 0.01
diag_tau_s = (1.0, 5.0)

# MO sim calibration (§3a-b → §7)
mo_self_scale = 0.7645
mo_impact_scale = 1.0
mo_cal_kw = dict(mo_self_scale=mo_self_scale, mo_impact_scale=mo_impact_scale)
mo_cal_run_fit = False       # True: 500k-event impact fit (~few min)
mo_cal_regen_full = False    # True: regen 1M sim days + adapted DB after fit
mo_cal_dir = resolve_data_path("mo_cal")

print(f"Empirical DB: {emp_db.name}  day={emp_day}")
print(f"gammas={len(sweep_gammas)}")
print(f"§3a diagnostic γ={diag_gamma}")